In [3]:
# ============================================================
# Model 4: CNN Text Classifier Baseline
# Terminal Table Output Only
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from collections import Counter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")

SEED = 42
LABEL_FRACTIONS = [0.05, 0.10, 0.20]

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# DATA READING AND PREPROCESSING
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    for enc in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception:
            pass
    raise ValueError(f"Cannot read {path}")

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "0", "false", "no"}:
            return 0
    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)
    return np.nan

URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s!?]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")

LAUGHTER_VARIANTS = {
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "hehe": "tertawa"
}

def basic_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\\n", " ").replace("\\/", "/")
    text = REPEAT_CHAR_PATTERN.sub(r"\1\1", text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(lambda m: f" {m.group(1)} ", text)
    text = text.lower()
    text = NON_ALNUM_PATTERN.sub(" ", text)
    return MULTISPACE_PATTERN.sub(" ", text).strip()

def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    c1, c2 = df.columns[:2]
    slang = {}
    for _, row in df.iterrows():
        src = str(row[c1]).strip().lower()
        tgt = str(row[c2]).strip().lower()
        if src and src != "nan":
            slang[src] = tgt
    return slang

def preprocess_text(text, slang_dict):
    text = basic_clean_text(text)
    tokens = []
    for tok in text.split():
        tok = LAUGHTER_VARIANTS.get(tok, tok)
        tok = slang_dict.get(tok, tok)
        tokens.append(tok)
    return " ".join(tokens)

def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label": "raw_label", "Tweet": "text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text", "label"]].dropna()

def load_572_dataset(path):
    df = safe_read_csv(path)
    text_col = next((c for c in ["comment_text", "tweet", "Tweet", "text", "Text"] if c in df.columns), None)
    label_col = next((c for c in ["class", "label", "Label", "HS"] if c in df.columns), None)
    if text_col is None or label_col is None:
        raise ValueError(f"Cannot detect text/label columns in {path}")
    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text", "label"]].dropna()

def load_re_dataset(path):
    df = safe_read_csv(path)
    text_col = next((c for c in ["Tweet", "tweet", "text", "Text"] if c in df.columns), None)
    label_col = "HS" if "HS" in df.columns else "label"
    if text_col is None or label_col is None:
        raise ValueError(f"Cannot detect columns in {path}")
    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text", "label"]].dropna()

def merge_datasets():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)

    data = pd.concat([
        load_idhsd_dataset(PATH_IDHSD),
        load_572_dataset(PATH_572),
        load_re_dataset(PATH_RE)
    ], ignore_index=True)

    data = data.drop_duplicates(subset=["text"]).reset_index(drop=True)
    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].reset_index(drop=True)

    print("=" * 70)
    print("FINAL DATASET SIZE :", len(data))
    print("LABEL DISTRIBUTION :", Counter(data["label"]))
    print("DEVICE             :", DEVICE)
    print("=" * 70)

    return data

# ============================================================
# DATASET AND MODEL
# ============================================================

def tokenize(text):
    return text.lower().split()

def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<pad>": 0, "<unk>": 1}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenize(self.texts[idx])[:self.max_len]
        ids = [self.vocab.get(tok, self.vocab["<unk>"]) for tok in tokens]

        if len(ids) == 0:
            ids = [self.vocab["<unk>"]]

        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

def collate_batch(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts, labels

class CNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 128, kernel_size=k, padding=0)
            for k in [3, 4, 5]
        ])

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128 * 3, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)

        conv_outputs = []
        for conv in self.convs:
            if x.size(2) < conv.kernel_size[0]:
                pad_size = conv.kernel_size[0] - x.size(2)
                x_padded = torch.nn.functional.pad(x, (0, pad_size))
                out = torch.relu(conv(x_padded))
            else:
                out = torch.relu(conv(x))

            out = torch.max(out, dim=2)[0]
            conv_outputs.append(out)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return self.fc(x)

# ============================================================
# TRAINING AND EVALUATION
# ============================================================

def predict_model(model, loader):
    model.eval()
    all_true, all_pred, all_prob = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = model(x)
            prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            pred = (prob >= 0.5).astype(int)

            all_true.extend(y.numpy())
            all_pred.extend(pred)
            all_prob.extend(prob)

    return np.array(all_true), np.array(all_pred), np.array(all_prob)

def find_best_threshold(y_true, y_prob):
    best_t = 0.5
    best_f1 = -1

    for t in np.arange(0.20, 0.81, 0.02):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t

def train_model(model, train_loader, val_loader, class_weights):
    model.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

    best_f1 = -1
    best_state = None

    for epoch in range(EPOCHS):
        model.train()

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        y_val, val_pred, _ = predict_model(model, val_loader)
        val_f1 = f1_score(y_val, val_pred, average="macro")

        print(f"Epoch {epoch + 1}/{EPOCHS} | Validation Macro-F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.to(DEVICE)

    return model

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

def run_experiment(train_df, val_df, test_df, frac):
    vocab = build_vocab(train_df["clean_text"].tolist(), min_freq=1)

    train_ds = TextDataset(train_df["clean_text"].tolist(), train_df["label"].astype(int).tolist(), vocab, MAX_LEN)
    val_ds = TextDataset(val_df["clean_text"].tolist(), val_df["label"].astype(int).tolist(), vocab, MAX_LEN)
    test_ds = TextDataset(test_df["clean_text"].tolist(), test_df["label"].astype(int).tolist(), vocab, MAX_LEN)

    class_counts = np.bincount(train_df["label"].astype(int).values, minlength=2)
    class_weights = torch.tensor(
        len(train_df) / (2.0 * np.maximum(class_counts, 1)),
        dtype=torch.float
    )

    sample_weights = [class_weights[int(y)].item() for y in train_df["label"].values]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate_batch)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_batch)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_batch)

    model = CNNTextClassifier(vocab_size=len(vocab))
    model = train_model(model, train_loader, val_loader, class_weights)

    y_val, _, val_prob = predict_model(model, val_loader)
    best_t = find_best_threshold(y_val, val_prob)

    y_test, _, test_prob = predict_model(model, test_loader)
    test_pred = (test_prob >= best_t).astype(int)

    result = evaluate_predictions(y_test, test_pred, test_prob)
    result["fraction"] = frac
    result["train_size"] = len(train_df)
    result["threshold"] = best_t
    result["vocab_size"] = len(vocab)

    return result

# ============================================================
# PRINT TABLE
# ============================================================

def print_results_table(results):
    print("\n" + "=" * 120)
    print("{:<10} {:<12} {:<12} {:<12} {:<10} {:<10} {:<10} {:<10} {:<10}".format(
        "Fraction", "TrainSize", "VocabSize", "Threshold", "Accuracy", "MacroF1", "Precision", "Recall", "ROC_AUC"
    ))
    print("=" * 120)

    for r in results:
        print("{:<10} {:<12} {:<12} {:<12.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
            r["fraction"],
            r["train_size"],
            r["vocab_size"],
            r["threshold"],
            r["accuracy"],
            r["macro_f1"],
            r["precision"],
            r["recall"],
            r["roc_auc"]
        ))

    print("=" * 120)

# ============================================================
# MAIN
# ============================================================

data = merge_datasets()

train_val_df, test_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    stratify=data["label"]
)

train_df_full, val_df = train_test_split(
    train_val_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_val_df["label"]
)

results = []

for frac in LABEL_FRACTIONS:
    print("\n" + "=" * 80)
    print(f"Running Model 4: CNN | Label Fraction = {frac}")
    print("=" * 80)

    labeled_train_df, _ = train_test_split(
        train_df_full,
        train_size=frac,
        random_state=SEED,
        stratify=train_df_full["label"]
    )

    res = run_experiment(labeled_train_df, val_df, test_df, frac)
    results.append(res)

print_results_table(results)

print("\nMODEL 4: CNN TEXT CLASSIFIER BASELINE FINISHED.")

FINAL DATASET SIZE : 13899
LABEL DISTRIBUTION : Counter({0.0: 7966, 1.0: 5933})
DEVICE             : cuda

Running Model 4: CNN | Label Fraction = 0.05
Epoch 1/50 | Validation Macro-F1: 0.4298
Epoch 2/50 | Validation Macro-F1: 0.5304
Epoch 3/50 | Validation Macro-F1: 0.6602
Epoch 4/50 | Validation Macro-F1: 0.5853
Epoch 5/50 | Validation Macro-F1: 0.6269
Epoch 6/50 | Validation Macro-F1: 0.6007
Epoch 7/50 | Validation Macro-F1: 0.5913
Epoch 8/50 | Validation Macro-F1: 0.6491
Epoch 9/50 | Validation Macro-F1: 0.6176
Epoch 10/50 | Validation Macro-F1: 0.6336
Epoch 11/50 | Validation Macro-F1: 0.6137
Epoch 12/50 | Validation Macro-F1: 0.6305
Epoch 13/50 | Validation Macro-F1: 0.6241
Epoch 14/50 | Validation Macro-F1: 0.6087
Epoch 15/50 | Validation Macro-F1: 0.6036
Epoch 16/50 | Validation Macro-F1: 0.6072
Epoch 17/50 | Validation Macro-F1: 0.6065
Epoch 18/50 | Validation Macro-F1: 0.6007
Epoch 19/50 | Validation Macro-F1: 0.6051
Epoch 20/50 | Validation Macro-F1: 0.6029
Epoch 21/50 | Val

In [2]:
import os
os.getcwd()

'/net/tscratch/people/plgshoffan/HateSpeech/paper_outputs_hybrid_plus/Experiment Used'